In [1]:
MODEL_PATH = "model.pkl"
FEATURE_ORDER_PATH = "feature_order.json"
OUTPUT_JSON_PATH = "rf_trees.json"

TRAIN_PATH = "train_set_selected.csv"
N_VALIDATION_SAMPLES = 1000

In [2]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [3]:
model = joblib.load(MODEL_PATH)
print(f"Model loaded: {type(model).__name__}")
print(f"  n_estimators : {model.n_estimators}")
print(f"  n_features   : {model.n_features_in_}")
print(f"  n_classes    : {len(model.classes_)}")
print(f"  classes      : {model.classes_}")

with open(FEATURE_ORDER_PATH, "r") as f:
    feature_order = json.load(f)

print(f"\nFeature order loaded: {len(feature_order)} fitur")
assert len(feature_order) == model.n_features_in_, \
    f"Jumlah fitur tidak cocok: feature_order ({len(feature_order)}) vs model ({model.n_features_in_})"

Model loaded: RandomForestClassifier
  n_estimators : 100
  n_features   : 33
  n_classes    : 2
  classes      : [0. 1.]

Feature order loaded: 33 fitur


In [4]:
trees_data = []

for i, estimator in enumerate(model.estimators_):
    tree = estimator.tree_

    tree_dict = {
        "children_left": tree.children_left.tolist(),
        "children_right": tree.children_right.tolist(),
        "threshold": [round(v, 6) for v in tree.threshold.tolist()],
        "feature": tree.feature.tolist(),
        "values": tree.value.tolist(),  # shape: [n_nodes, 1, n_classes]
    }

    trees_data.append(tree_dict)

# Hitung total nilai leaf
total_leaves = sum(sum(1 for cl in t["children_left"] if cl == -1) for t in trees_data)

print(f"Jumlah trees diekstrak: {len(trees_data)}")
print(f"Total leaf nodes: {total_leaves}")
print(f"Rata-rata leaf per tree: {total_leaves / len(trees_data):.1f}")
print(f"\nContoh struktur tree ke-0 (10 node pertama):")
t0 = trees_data[0]
print(f"  children_left[:10] : {t0['children_left'][:10]}")
print(f"  children_right[:10]: {t0['children_right'][:10]}")
print(f"  threshold[:10]     : {t0['threshold'][:10]}")
print(f"  feature[:10]       : {t0['feature'][:10]}")

Jumlah trees diekstrak: 100
Total leaf nodes: 246936
Rata-rata leaf per tree: 2469.4

Contoh struktur tree ke-0 (10 node pertama):
  children_left[:10] : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  children_right[:10]: [2494, 1127, 1126, 471, 378, 33, 32, 31, 28, 11]
  threshold[:10]     : [1.5, -0.5, 10.5, 1.5, 0.5, 9.5, 4.5, 0.5, 3.5, 1.5]
  feature[:10]       : [0, 21, 5, 11, 16, 4, 32, 25, 32, 32]


In [5]:
rf_json = {
    "n_features": model.n_features_in_,
    "n_classes": len(model.classes_),
    "n_estimators": model.n_estimators,
    "feature_names": feature_order,
    "trees": trees_data,
}

# Simpan ke file
with open(OUTPUT_JSON_PATH, "w") as f:
    json.dump(rf_json, f)

file_size = joblib.os.path.getsize(OUTPUT_JSON_PATH)
print(f"rf_trees.json disimpan: {OUTPUT_JSON_PATH}")
print(f"Ukuran file: {file_size / 1024 / 1024:.2f} MB")

rf_trees.json disimpan: rf_trees.json
Ukuran file: 22.69 MB


In [6]:
def predict_rf_trees(rf_json, features):
    """
    Predict dengan struktur JSON trees.
    Logika ini akan diterjemahkan 1:1 ke JavaScript.
    """
    n_classes = rf_json["n_classes"]
    n_estimators = rf_json["n_estimators"]
    scores = np.zeros(n_classes)

    for tree in rf_json["trees"]:
        node = 0
        while tree["children_left"][node] != -1:
            if features[tree["feature"][node]] <= tree["threshold"][node]:
                node = tree["children_left"][node]
            else:
                node = tree["children_right"][node]
        scores += tree["values"][node][0]

    proba = scores / n_estimators
    return proba

In [8]:
# Load data untuk validasi
df_val = pd.read_csv(TRAIN_PATH).sample(N_VALIDATION_SAMPLES, random_state=42)

# Cari target col
candidates = ["label", "Label", "class", "Class", "type", "Type",
             "target", "Target", "phishing", "Phishing", "y", "Result", "result"]
target_col = None
for c in candidates:
    if c in df_val.columns:
        target_col = c
        break
assert target_col is not None, "Target column not found"

X_val = df_val.drop(columns=[target_col]).values
y_val = df_val[target_col].values

print(f"Data validasi: {len(X_val)} sampel, {X_val.shape[1]} fitur")

Data validasi: 1000 sampel, 33 fitur


In [9]:
# Bandingkan prediksi
pred_sklearn = model.predict_proba(X_val)

pred_json_list = []
for i in range(len(X_val)):
    pred = predict_rf_trees(rf_json, X_val[i])
    pred_json_list.append(pred)
pred_json = np.array(pred_json_list)

# Bandingkan
diff = np.max(np.abs(pred_sklearn - pred_json))
match = np.allclose(pred_sklearn, pred_json, atol=1e-10)

print("HASIL VALIDASI")
print(f"Sampel divalidasi: {len(X_val)}")
print(f"Maksimum perbedaan probabilitas: {diff:.2e}")
print(f"Semua prediksi identik? {'YA' if match else 'GA'}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


HASIL VALIDASI
Sampel divalidasi: 1000
Maksimum perbedaan probabilitas: 2.22e-16
Semua prediksi identik? YA


In [10]:
# Bandingkan kelas hasil prediksi (hard voting)
class_sklearn = model.predict(X_val)
class_json = (pred_json[:, 1] > 0.5).astype(int)

accuracy_validation = (class_sklearn == class_json).mean()
print(f"Kecocokan kelas prediksi: {accuracy_validation:.4f}")

if accuracy_validation == 1.0:
    print("DONE BANG!")
else:
    mismatch = np.where(class_sklearn != class_json)[0]
    print(f"Ada {len(mismatch)} sampel yang tidak cocok.")
    print(f"Contoh mismatch index: {mismatch[:5]}")

Kecocokan kelas prediksi: 1.0000
DONE BANG!


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
